# Post-Cavity Experiment — Context Mode

This notebook demonstrates the **context-mode** workflow where sessions are
scoped to a specific `device_id` + `cooldown_id`. Calibrations are automatically
bound to the hardware context, preventing stale-calibration reuse across different
devices or cooldowns.

## Prerequisites

- qubox_v2 installed
- Existing `seq_1_device/config/` directory with device-level configs
- OPX+ hardware available (or run in simulation mode)

## 1. Set Up Device Registry

First, create a device in the registry using the existing `seq_1_device` config
files as the source.

In [ ]:
from pathlib import Path
from qubox_v2.devices import DeviceRegistry, DeviceInfo

# Registry root — where devices/ directory will live
REGISTRY_BASE = Path(".")  # adjust to your workspace root

registry = DeviceRegistry(REGISTRY_BASE)
print(f"Registry base: {registry.base_path}")
print(f"Existing devices: {registry.list_devices()}")

In [ ]:
DEVICE_ID = "post_cavity_sample_A"

if not registry.device_exists(DEVICE_ID):
    dev_path = registry.create_device(
        DEVICE_ID,
        description="Transmon qubit A — 3D cavity sample",
        config_source=Path("seq_1_device/config"),  # copy hardware.json, cqed_params.json, etc.
        sample_info={"chip": "Q1-2025A", "fridge": "BlueFors-LD400"},
    )
    print(f"Created device at: {dev_path}")
else:
    print(f"Device '{DEVICE_ID}' already exists")

info = registry.load_device_info(DEVICE_ID)
print(f"Device: {info.device_id} — {info.description}")

## 2. Create a Cooldown

Each cooldown cycle gets its own calibration, pulse, and measurement config.
Optionally seed from the previous cooldown's config.

In [ ]:
COOLDOWN_ID = "cd_2025_02_22"

if not registry.cooldown_exists(DEVICE_ID, COOLDOWN_ID):
    # Optionally seed from existing seq_1_device/config for initial pulses.json etc.
    cd_path = registry.create_cooldown(
        DEVICE_ID, COOLDOWN_ID,
        seed_from=Path("seq_1_device/config"),
    )
    print(f"Created cooldown at: {cd_path}")
else:
    print(f"Cooldown '{COOLDOWN_ID}' already exists")

print(f"Cooldowns for {DEVICE_ID}: {registry.list_cooldowns(DEVICE_ID)}")

## 3. Open Session in Context Mode

Pass `device_id` and `cooldown_id` instead of `experiment_path`.
The session resolves all config paths automatically:
- Device-level: hardware.json, cqed_params.json, devices.json
- Cooldown-level: calibration.json, pulses.json, measureConfig.json

In [ ]:
from qubox_v2.experiments.session import SessionManager

session = SessionManager(
    device_id=DEVICE_ID,
    cooldown_id=COOLDOWN_ID,
    registry_base=REGISTRY_BASE,
    qop_ip="10.0.0.1",  # adjust to your OPX+ IP
    auto_save_calibration=True,
)

# Verify context is populated
ctx = session.context
print(f"Device ID:    {ctx.device_id}")
print(f"Cooldown ID:  {ctx.cooldown_id}")
print(f"Wiring Rev:   {ctx.wiring_rev}")
print(f"Config Hash:  {ctx.config_hash}")
print(f"Schema Ver:   {ctx.schema_version}")

In [ ]:
# Open QM connection
session.open()

# Verify calibration context is stamped
cal_ctx = session.calibration.data.context
if cal_ctx:
    print(f"Calibration bound to device: {cal_ctx.device_id}")
    print(f"Calibration bound to cooldown: {cal_ctx.cooldown_id}")
    print(f"Calibration wiring rev: {cal_ctx.wiring_rev}")
else:
    print("No context block in calibration (legacy v3 file)")

## 4. Run Calibration Pipeline

The calibration pipeline is identical to legacy mode — experiment classes
are completely device-agnostic. Only the session infrastructure changed.

In [ ]:
# Example: Readout discrimination calibration
from qubox_v2.experiments.calibration.readout import ReadoutGEDiscrimination

disc_exp = ReadoutGEDiscrimination(session)
# result = disc_exp.run(n_total=10000)
# analysis = disc_exp.analyze(result, update_calibration=True)
# disc_exp.plot(analysis)
print("Readout discrimination experiment ready (uncomment to run)")

In [ ]:
# Example: Rabi experiment
from qubox_v2.experiments.time_domain.rabi import PiAmplitudeCalibration

rabi_exp = PiAmplitudeCalibration(session)
# result = rabi_exp.run(n_total=5000)
# analysis = rabi_exp.analyze(result, update_calibration=True)
# rabi_exp.plot(analysis)
print("Pi amplitude calibration experiment ready (uncomment to run)")

## 5. Verify Context Stamped in Calibration

After running calibrations, the calibration.json file should have the context
block embedded, binding it to this specific device and cooldown.

In [ ]:
import json

cal_path = session.calibration.path
print(f"Calibration file: {cal_path}")
print(f"Calibration version: {session.calibration.data.version}")

# Show the context block
cal_dict = session.calibration.to_dict()
if cal_dict.get("context"):
    print("\nContext block:")
    print(json.dumps(cal_dict["context"], indent=2))
else:
    print("\nNo context block (will be stamped on first save)")

# Show calibration summary
print("\n" + session.calibration.summary())

## 6. Context Mismatch Protection

If you try to open calibration from a different device, the system raises
`ContextMismatchError` (in strict mode) or warns (in non-strict mode).

In [ ]:
from qubox_v2.core.errors import ContextMismatchError
from qubox_v2.core.experiment_context import ExperimentContext
from qubox_v2.calibration.store import CalibrationStore

# Simulate a mismatched context
wrong_ctx = ExperimentContext(
    device_id="different_device",
    cooldown_id=COOLDOWN_ID,
    wiring_rev="00000000",
)

try:
    bad_store = CalibrationStore(
        cal_path,
        context=wrong_ctx,
        strict_context=True,
    )
except ContextMismatchError as e:
    print(f"Context mismatch detected: {e}")
else:
    print("No mismatch (calibration file may not have a context block yet)")

## 7. Close Session

In [ ]:
session.close()
print("Session closed.")

## Alternative: Legacy Mode (Backward Compatible)

The old `experiment_path` mode still works exactly as before:

```python
session = SessionManager("./seq_1_device", qop_ip="10.0.0.1")
session.open()
print(session.context)  # None — legacy mode
session.close()
```